# CSCI 6032: Homework 2

## Working Safely with Agents, Git, Docker, and Skills

This homework is about **using AI agents effectively and safely**. It is not about how a language model works internally. You will install GitHub Copilot CLI, give an agent a bounded workspace, use Git checkpoints before delegating changes, construct a reproducible Linux development container, run an agent inside that container, and create a reusable skill that helps submit the completed homework.

The notebook is the spine of your submission. Treat it as a laboratory notebook: record what you tried, what happened, false starts, corrections, and what you learned. The code, tests, `Dockerfile`, agent instructions, and skill must remain as ordinary files in the repository.

**Due: Thursday, September 17, 2026, 11:00 PM.**

## Learning objectives

After completing this homework, you should be able to:

1. Install, authenticate, and run a command-line coding agent.
2. Explain the difference between instructions, permissions, and an operating-system sandbox.
3. Restrict an agent to a designated project directory.
4. Use Git commits as recovery checkpoints before and after agent work.
5. Inspect and test an agent's changes rather than accepting them blindly.
6. Build and run a small Linux development environment with Docker.
7. Bind-mount a host repository into a container without exposing unrelated host files.
8. Run GitHub Copilot CLI inside the container and verify that changes persist on the host.
9. Package a repeated workflow as an agent skill.
10. Use browser automation with explicit human approval at an irreversible step.

## Grading and use of AI

You may use AI in any way you find useful on this homework. You remain responsible for every command you approve, every file you submit, and every claim in this notebook.

This homework is graded primarily on **completeness; reproducible evidence; and thoughtful reflection**. A failed attempt can be valuable evidence when you explain the failure and how you diagnosed it. 

Never present an agent's statement as verification. Verification means that you inspected the relevant state yourself—for example, with `git status`, `git diff`, a test command, a file listing, or a Blackboard confirmation page.

## Safety rules

These rules apply throughout the assignment.

- Never place passwords, API keys, access tokens, private keys, authentication cookies, browser profiles, or Blackboard credentials in the repository, notebook, prompts, Dockerfile, image, skill, or screenshots.
- Redact usernames, email addresses, tokens, device codes, and other private information from captured output.
- Launch the host agent from the homework repository—not from your home directory or a broad parent directory.
- Read each requested command before approving it. Deny a command you do not understand and ask the agent to explain.
- Do not use destructive Git commands, rewrite history, or force-push.
- Do not mount your home directory, `.ssh`, or Docker socket into the container.
- Authenticate interactively. Do not put credentials in the Dockerfile or bake them into an image layer.  You can login using the browser and then let copilot drive from there when you want to automate the web.
- You must personally authenticate to Blackboard.

If you accidentally expose a credential, revoke it. Removing it in a later commit does not remove it from Git history.  You can rewrite git history, but it must be done explicitly.

## What you will submit

You will create a GitHub repository named:

```text
csci6032-hw2-<your-github-username>
```

Clearly anyone can look at this repository.  That is fine.

The repository will contain this completed notebook and the supporting artifacts. At the end, your submission skill will prepare a compressed archive from a clean, committed revision and use your authenticated browser session to help submit:

1. `CSCI6032_hw2.ipynb`
2. `csci6032-hw2-<your-github-username>.tar.gz`
3. The URL of the public GitHub repository.

Do not include the repository's `.git` directory in the archive. The public GitHub repository preserves the Git history.

If publishing your repository would expose information that should not be public, contact the instructor before publishing it.

## Part 0. Access and preparation (5 points)

Use your own GitHub account. If you do not already have Copilot access, request the GitHub Education student benefit immediately. Approval can take time. GitHub Copilot CLI is available with current Copilot plans; consult the instructor if account approval becomes a blocker.

Related links:

- [GitHub Education](https://github.com/education)
- [Install GitHub Copilot CLI](https://docs.github.com/en/copilot/how-tos/copilot-cli/set-up-copilot-cli/install-copilot-cli)
- [Authenticate GitHub Copilot CLI](https://docs.github.com/en/copilot/how-tos/copilot-cli/set-up-copilot-cli/authenticate-copilot-cli)

Record:

- your host operating system and version;
- the date you requested or confirmed Copilot access;
- whether access is active;
- any setup obstacle and how you addressed it.

Do not include personal documents or screenshots from the student-verification process.

### Part 0 laboratory record

**Host operating system and version:**

**Answer**

Windows 11 Home, version 25H2, OS Build 26200.9457

**Copilot access status and date:**

**Answer**

Yeah I have just got the access and installed copilot in my homework repo.

**Setup notes, including any false starts:**

**Answer**

I created a new repo in my github profile and cloned it in my C drive. Then I will put the homework file into my homework repo. 

## Part 1. Install and run GitHub Copilot CLI on the host (10 points)

Install the current stable GitHub Copilot CLI using one of the methods in the official documentation. Record the method and version; do not assume that commands copied from an old tutorial remain current.

After installation:

1. Run `copilot version`.
2. Run `copilot login` or start `copilot` and use `/login`.
3. Complete the interactive browser or device-code flow yourself.
4. Start an interactive session and enter this prompt:

```text
Reply with exactly: Hello, CSCI 6032.
```

5. Exit the agent cleanly.

This is the agent equivalent of a Hello World exercise. It verifies access before the assignment introduces files, tools, and permissions.

### Part 1 laboratory record

Record the installation method, the redacted output of `copilot version`, and the Hello World response. Do not paste authentication codes, tokens, or account details.

**Answer**

(base) PS C:\csci6032-hw2-batorgil7> npm install -g @github/copilot

added 3 packages in 1m
npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.2
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.2
npm notice To update run: npm install -g npm@12.0.2
npm notice
(base) PS C:\csci6032-hw2-batorgil7> copilot version
GitHub Copilot CLI 1.0.87

You are running the latest version.
Folder C:\csci6032-hw2-batorgil7 has been added to trusted folders.
Reply with exactly: Hello, CSCI 6032
Hello, CSCI 6032

What, if anything, did the model need permission to do for the Hello World prompt? Why?

**Answer**

The model asked me if I'm trusting the folder first before I send any message to it. It's because copilot can edit and view my files in my folder. So first I need to give permission that it can access and edit my files. Basically I'm giving the agent a permission to edit my files.

## Part 2. Create the repository and its first checkpoint (10 points)

On GitHub, create the public repository named `csci6032-hw2-<your-github-username>`, initialize it with a `README.md`, and clone it onto your host computer.

Copy this notebook into the repository root as `CSCI6032_hw2.ipynb`. Add a small UTF-8 text file named `sample.txt` containing at least five lines of text that you wrote yourself. Update `README.md` with the course, homework title, repository URL, your host operating system, and a short description.

Before invoking an agent on the repository, inspect the state:

```bash
git remote -v
git status
git diff
```

Commit and push these initial files. Use this commit message:

```text
checkpoint: repository before agent changes
```

Create and switch to a branch named `agent-work` before continuing. Confirm that the working tree is clean.

### Part 2 laboratory record

Paste a concise, redacted record showing:

- the repository URL;
- the initial checkpoint commit identifier;
- the current branch;
- a clean `git status` after the checkpoint.

**Answer**

- Repository URL: https://github.com/BatOrgil7/csci6032-hw2-batorgil7
- Initial checkpoint commit: `639b591` ("checkpoint: repository before agent changes")
- Current branch: `agent-work`
- Working tree: clean

```text
(base) PS C:\csci6032-hw2-batorgil7> git remote -v
origin  https://github.com/BatOrgil7/csci6032-hw2-batorgil7.git (fetch)
origin  https://github.com/BatOrgil7/csci6032-hw2-batorgil7.git (push)

(base) PS C:\csci6032-hw2-batorgil7> git add .
warning: in the working copy of 'CSCI6032_hw2.ipynb', LF will be replaced by CRLF the next time Git touches it
(base) PS C:\csci6032-hw2-batorgil7> git commit -m "checkpoint: repository before agent changes"
[main 639b591] checkpoint: repository before agent changes
 1 file changed, 1 insertion(+), 1 deletion(-)

(base) PS C:\csci6032-hw2-batorgil7> git switch -c agent-work
Switched to a new branch 'agent-work'
(base) PS C:\csci6032-hw2-batorgil7> git status
On branch agent-work
nothing to commit, working tree clean

(base) PS C:\csci6032-hw2-batorgil7> git push origin main
To https://github.com/BatOrgil7/csci6032-hw2-batorgil7.git
   0536b16..639b591  main -> main
```

**False starts:**

- I typed `gi add .` instead of `git add .`. In PowerShell, `gi` is an alias for `Get-Item`, so it failed with a parameter error. I retyped it as `git`.
- My first `git commit` said "nothing to commit" because my notebook edits hadn't been saved yet. After saving, `git add .` and `git commit` worked.
- I tried `git new branch`, which isn't a Git command. Then `git branch create` made a branch literally named `create`. I switched to the correct branch with `git switch -c agent-work` and removed the mistaken one with `git branch -d create`. It pointed at the same commit (`639b591`), so no work was lost.
- The commit landed on `main` locally before I pushed it. I pushed it with `git push origin main`, which moved `origin/main` from `0536b16` to `639b591`.

Why is a commit more useful than merely copying a few files before an agent works?

**Answer**

A commit captures a snapshot of the entire repository, not just the few files I think the agent will touch. Agents often edit, create, or delete files I didn't expect, and a manual copy would miss those. With a commit I can run `git status` and `git diff` to see exactly what the agent changed, line by line. I can also go back with `git restore` or `git revert`, or throw away the agent's branch entirely. Because the commit is pushed to GitHub, the checkpoint also exists outside my machine, so a mistake on my computer can't destroy it. It also has an ID (`639b591`) and a message, so later work can be compared to a clear, named starting point. Copied files have no history, no diff tooling, and no record of when or why they were saved.

## Part 3. Constrain the host agent (10 points)

Create `AGENTS.md` at the root of the repository. You may ask an agent to draft it, but you must inspect and revise the result. It must instruct agents to:

1. Work only inside the current repository.
2. Never read, print, store, commit, or upload secrets, credentials, private keys, browser data, or configuration files that may contain them.
3. Explain an intended change before editing.
4. Ask before installing software, accessing a new network destination, deleting files, changing Git history, committing, or pushing.
5. Preserve uncommitted user work and avoid destructive Git commands and force-pushes.
6. Check `git status` before editing and stop if unrelated changes are present.
7. Make small, reviewable changes.
8. Show `git diff` after editing and run the smallest relevant test.
9. Explain errors rather than silently ignoring them.
10. Never claim success without checking the requested result.

Start Copilot CLI from the repository root. Enable local sandboxing when it is available, then inspect its actual state and policy with:

```text
/sandbox enable
/sandbox status
/sandbox policy
```

The current working directory should be writable; unrelated host locations should not be writable. Do not add broad path grants. If local sandboxing is unavailable on your platform or version, document that result and rely on the repository boundary, permission prompts, and the Docker phase that follows.

Commit `AGENTS.md` with the message `docs: add agent safety instructions` and push the branch.

### Part 3 laboratory record

Summarize the effective sandbox policy without exposing personal paths. State whether sandboxing was active and whether you changed any default permissions.

**Answer**

Local sandboxing was **not** available and therefore not active. Copilot CLI 1.0.87 on Windows 11 does not recognize the command at all:

```text
> /sandbox enable
Unknown command: /sandbox
```

`/sandbox status` and `/sandbox policy` fail the same way, and `/sandbox` does not appear in `/help`. I did not change any default permissions and added no path grants.

Because there is no sandbox, nothing at the operating-system level prevented the agent from reading or writing outside the repository. The effective boundary was therefore a combination of conventions rather than enforcement: I launched Copilot from the repository root so the trusted folder was the project directory, `AGENTS.md` instructed the agent to stay inside the repository, and I reviewed each permission prompt before approving it. That is meaningfully weaker than a real sandbox, since an agent that ignored those instructions would not have been stopped. The Docker container in Parts 6 and 7 supplies the enforced boundary that was missing here.

Explain why `AGENTS.md` is useful but is not, by itself, a security boundary.

**Answer**

`AGENTS.md` is useful because it gives the agent a clear, durable set of instructions about scope, safety, Git hygiene, and verification. It helps standardize behavior and reduces accidental drift, especially when a model is allowed to edit files or run commands.

It is not a security boundary by itself because it is only a text file. The operating system, Git permissions, the trusted-directory setting, and user approval still determine what the agent can actually access or change. If a model is running with broad filesystem access, an `AGENTS.md` file can be ignored or not enforced. The real protection comes from sandboxing, restricted paths, permission prompts, and careful human review.


## Part 4. Generate and review code with the host agent (15 points)

Begin with a clean working tree on `agent-work`. Ask the host agent to complete this bounded task:

```text
Create a small Python command-line program in src/text_stats.py. It must read
one UTF-8 text file and print JSON containing the number of lines, words, and
characters. Add tests using Python's built-in unittest framework, document the
usage in README.md, and test the program with sample.txt. Do not add external
Python dependencies. Do not commit or push. Before editing, explain your plan;
after editing, show the diff and run the tests.
```

You may clarify the prompt or help the agent recover from errors. Do not ask it to work outside the repository.

When it finishes:

1. Run `git status` and `git diff` yourself.
2. Read every changed file.
3. Run the tests yourself.
4. Run the program on `sample.txt` and independently check at least one reported count.
5. Correct any problem, recording whether you or the agent made the correction.
6. Commit the reviewed result as `feat: add agent-generated text statistics tool`.
7. Push `agent-work`.

### Part 4 laboratory record

Record the exact prompt you used if it differs from the supplied prompt. Include selected agent output that shows planning, tool use, and verification; do not paste an enormous transcript.

**Answer**

I used the supplied prompt unchanged.

```text
Create a small Python command-line program in src/text_stats.py. It must read
one UTF-8 text file and print JSON containing the number of lines, words, and
characters. Add tests using Python's built-in unittest framework, document the
usage in README.md, and test the program with sample.txt. Do not add external
Python dependencies. Do not commit or push. Before editing, explain your plan;
after editing, show the diff and run the tests.
```

Selected agent output included:

```text
I'll first inspect the repository status and existing project structure, then add the
CLI, focused `unittest` coverage, README usage, and a UTF-8 sample fixture.
The worktree is clean, and the repository has no existing Python test/configuration
structure...
The sample invocation established the fixture's actual totals as 5 lines, 40 words,
and 227 characters, so the tests and README now match the requested program behavior.
All four tests pass, and the sample command prints the documented JSON.
```

Record the test command and result, the independent check you performed, and the commit identifier.

**Answer**

```bash
python -m unittest discover -s tests -v
```

Result: 4 tests ran and all 4 passed.

Independent check:

```bash
python src/text_stats.py sample.txt
```

Output: `{"lines": 5, "words": 40, "characters": 227}`

I also verified the counts directly from `sample.txt` with a separate Python one-liner that read the file and recomputed `len(text.splitlines())`, `len(text.split())`, and `len(text)`, rather than trusting the program's own report. All three matched.

Commit identifier: `18cef47`

What did you inspect before deciding that the change was safe to commit? Did the agent do anything unexpected?

**Answer**

Before committing, I inspected `git status`, reviewed the diff for each created or changed file, and read every edited file (`README.md`, `src/text_stats.py`, and `tests/test_text_stats.py`). I confirmed the unit tests and the sample CLI invocation matched the expected counts. The agent did not do anything unexpected; it created only the requested Python program, tests, and usage notes. The only issue was a transient shell permission warning during output rendering, but the actual commands completed successfully and the repository changes were then reviewed and committed.


## Part 5. Install and verify Docker on the host (10 points)

Install Docker Desktop or Docker Engine using the current official instructions for your operating system:

- [Docker Desktop overview and platform installers](https://docs.docker.com/desktop/)
- [Docker Engine installation](https://docs.docker.com/engine/install/)

Start Docker and verify it from a host terminal:

```bash
docker version
docker run --rm hello-world
```

The first command should show both client and server information. The second should retrieve and run a small test image successfully.

If institutional or hardware restrictions prevent a normal installation, contact the instructor early rather than weakening the assignment's safety requirements.

### Part 5 laboratory record

Record the Docker product, installation method, version, and a concise excerpt showing that `hello-world` ran. Note any virtualization, WSL, permission, or architecture issue you encountered.

**Answer**

- **Product:** Docker Desktop 4.85.0 (235549) for Windows
- **Installation method:** official Docker Desktop installer for Windows from docs.docker.com
- **Version:** Docker client 29.6.2, Docker Engine 29.6.2 (API 1.55)
- **Architecture:** Windows client `windows/amd64`, and the engine runs a Linux VM (`linux/amd64`, context `desktop-linux`)

**False start: the engine wasn't running.** My first `docker version` showed only the client section, then failed:

```text
(base) PS C:\csci6032-hw2-batorgil7> docker version
Client:
 Version:           29.6.2
 ...
 Context:           desktop-linux
failed to connect to the docker API at npipe:////./pipe/dockerDesktopLinuxEngine; check if the path is correct and if the daemon is running: open //./pipe/dockerDesktopLinuxEngine: The system cannot find the file specified.
```

`docker run --rm hello-world` failed with the same error. The Docker CLI was installed, but the Docker Desktop app, which starts the Linux engine, wasn't open. So the client had no daemon to connect to through the named pipe. I opened Docker Desktop, waited for "Engine running", and ran both commands again.

**After starting Docker Desktop:**

```text
(base) PS C:\csci6032-hw2-batorgil7> docker version
Client:
 Version:           29.6.2
 API version:       1.55
 OS/Arch:           windows/amd64
 Context:           desktop-linux

Server: Docker Desktop 4.85.0 (235549)
 Engine:
  Version:          29.6.2
  API version:      1.55 (minimum version 1.40)
  OS/Arch:          linux/amd64
 containerd:
  Version:          v2.2.5
 runc:
  Version:          1.3.6

(base) PS C:\csci6032-hw2-batorgil7> docker run --rm hello-world
Unable to find image 'hello-world:latest' locally
latest: Pulling from library/hello-world
Status: Downloaded newer image for hello-world:latest

Hello from Docker!
This message shows that your installation appears to be working correctly.
```

Both the client and server sections now appear, and `hello-world` was pulled from Docker Hub and ran successfully. Apart from the engine not running at first, I had no virtualization, WSL, permission, or architecture problems.

## Part 6. Create a minimal Linux agent development image (15 points)

Use an agent to help create a `Dockerfile`, but inspect and understand every instruction. The image must provide:

- a small, current Linux base suitable for Python development;
- Python 3.12 or later and `pip`;
- Git;
- GitHub CLI (`gh`);
- the current stable GitHub Copilot CLI;
- certificates and only the additional utilities needed for installation;
- a non-root development user;
- `/workspace` as the working directory.

The image must **not** contain your repository, tokens, credentials, browser profile, Docker socket, or a copy of a host credential directory. Do not use a floating prerelease of Copilot CLI. Prefer a reproducible version argument or record the resolved Copilot version.

Build the image with the tag:

```text
csci6032-hw2-agent
```

Start an interactive container with only the homework repository bind-mounted read/write at `/workspace`. Docker recommends the explicit `--mount` syntax. Have your agent adapt the following pattern to your shell and operating system:

```text
docker run --rm -it --mount type=bind,src=<ABSOLUTE-PATH-TO-REPOSITORY>,dst=/workspace csci6032-hw2-agent
```

Do not mount a broad parent directory. On native Linux, you may need to account for the host user's UID and GID so the non-root container user can write to the mounted repository; document any adjustment.

Inside the container, verify:

```bash
id
pwd
python --version
git --version
gh --version
copilot version
git status
```

Authenticate `gh` and Copilot interactively from inside the container using their browser/device flows. Do not paste a token into a Docker command, Dockerfile, notebook, or shell history. Demonstrate GitHub access with a read-only command such as `gh repo view`; redact account details in the notebook.

### Part 6 laboratory record

Paste your complete `Dockerfile` here as a fenced code block. Explain the purpose of each major layer and how the non-root user is created.

**Answer**

```Dockerfile
FROM ubuntu:24.04

ENV DEBIAN_FRONTEND=noninteractive
WORKDIR /workspace

RUN apt-get update && apt-get install -y --no-install-recommends \
    ca-certificates \
    curl \
    git \
    gnupg \
    lsb-release \
    python3 \
    python3-pip \
    npm \
    && rm -rf /var/lib/apt/lists/*

RUN mkdir -p /etc/apt/keyrings \
    && curl -fsSL https://cli.github.com/packages/githubcli-archive-keyring.gpg | gpg --dearmor -o /etc/apt/keyrings/githubcli-archive-keyring.gpg \
    && echo "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/githubcli-archive-keyring.gpg] https://cli.github.com/packages stable main" > /etc/apt/sources.list.d/github-cli.list \
    && apt-get update \
    && apt-get install -y --no-install-recommends gh \
    && rm -rf /var/lib/apt/lists/*

RUN npm install -g @github/copilot@1.0.87 \
    && ln -sf /usr/bin/python3 /usr/bin/python

RUN useradd --create-home --shell /bin/bash appuser \
    && mkdir -p /workspace /home/appuser \
    && chown -R appuser:appuser /workspace /home/appuser

USER appuser
RUN git config --global --add safe.directory /workspace
WORKDIR /workspace
```

The image starts from Ubuntu to provide a clean Linux base with a standard package manager and predictable Python toolchain. The first install step adds the utilities needed for development work: Git, Python 3, Node/npm, certificates, and curl. The GitHub CLI step adds the official `gh` repository and installs `gh` so the container can use GitHub from inside the container without bundling credentials. The Copilot step installs the stable CLI tool used for the assignment. The final user-creation step creates a non-root user (`appuser`) and ensures `/workspace` is writable for that account; setting `git config --global --add safe.directory /workspace` avoids false ownership errors when the mounted host repository is shared into the container. The container then runs as `appuser` in `/workspace`, so the working directory is a narrow, explicit mount rather than the entire host filesystem.


Record the exact build and run commands with private path components replaced by a placeholder. Include concise, redacted verification output from inside the container.

**Answer**

```powershell
# build (run from the repository root on the host)
docker build -t csci6032-hw2-agent .

# run an interactive container with only the repository bind-mounted
docker run --rm -it --mount type=bind,src=<ABSOLUTE-PATH-TO-REPOSITORY>,dst=/workspace csci6032-hw2-agent bash
```

Verification from inside the container (account details redacted):

```text
appuser@<container-id>:/workspace$ id
uid=1001(appuser) gid=1001(appuser) groups=1001(appuser)
appuser@<container-id>:/workspace$ pwd
/workspace
appuser@<container-id>:/workspace$ python --version
Python 3.12.3
appuser@<container-id>:/workspace$ git --version
git version 2.43.0
appuser@<container-id>:/workspace$ gh --version
gh version 2.101.0 (2026-09-15)
appuser@<container-id>:/workspace$ copilot version
GitHub Copilot CLI 1.0.87
You are running the latest version.
appuser@<container-id>:/workspace$ git status
On branch agent-work
Changes not staged for commit:
        modified:   .gitignore
        modified:   AGENTS.md
        modified:   CSCI6032_hw2.ipynb
        modified:   README.md
        modified:   sample.txt
        modified:   src/text_stats.py
        modified:   tests/test_text_stats.py

Untracked files:
        Dockerfile
```

The container runs as the non-root user `appuser` in `/workspace`, and the mounted repository is on the correct branch, so the bind mount reached the right host directory.

**Line-ending artifact.** Inside the container `git status` listed seven tracked files as modified, although the host working tree was clean apart from the notebook and the new `Dockerfile`. These were not agent edits. My host Git has `core.autocrlf=true`, so the working-tree files are CRLF on Windows while the committed blobs are LF. Linux Git in the container does no such conversion, so it reports every line as changed. I confirmed this with `git diff --stat AGENTS.md` inside the container, which reported 12 insertions and 12 deletions for a 12-line file: every line "changed," with no real content difference.

**Interactive authentication inside the container.** I ran `gh auth login` and chose GitHub.com, HTTPS, and "Login with a web browser." The container has no browser or clipboard tool, so `gh` could not open a page itself:

```text
! Failed to copy one-time code to clipboard
  No clipboard utilities available...
! First copy your one-time code: <REDACTED-DEVICE-CODE>
Press Enter to open https://github.com/login/device in your browser...
! Failed opening a web browser at https://github.com/login/device
  exec: "xdg-open,x-www-browser,www-browser,wslview": executable file not found in $PATH
  Please try entering the URL in your browser manually
✓ Authentication complete.
✓ Configured git protocol
! Authentication credentials saved in plain text
✓ Logged in as <REDACTED-GITHUB-USERNAME>
```

I opened the device URL in Chrome on the host myself and entered the one-time code there, so no token was ever typed into a Docker command, the Dockerfile, the notebook, or shell history. The credential `gh` stored lives inside the container's own filesystem, and because the container runs with `--rm` it is discarded when the container exits. It is not in the repository or the image.

Read-only GitHub access check:

```text
appuser@<container-id>:/workspace$ gh repo view
<REDACTED-GITHUB-USERNAME>/csci6032-hw2-batorgil7
...README contents for this homework repository...
View this repository on GitHub: https://github.com/<REDACTED-GITHUB-USERNAME>/csci6032-hw2-batorgil7
```

I then started `copilot` from `/workspace` inside the container. It reported `Folder /workspace has been added to trusted folders`, which confirms that the repository directory, not a broad host path, is the agent's working boundary.

What host resources can the container modify through the bind mount? What important resources did you deliberately not mount?

**Answer**

The container can modify only the files reachable through the explicit bind mount at `/workspace`: the homework repository itself. That means it can read and write the repository files, create or change files within that directory, and run tests or edits inside that project tree.

I deliberately did not mount the host home directory, `.ssh`, Docker socket, browser profile, or any broad parent directory. This keeps the container from reaching unrelated host content, tokens, or credential material. The repo boundary plus the narrow mount is the security boundary here, while the non-root `appuser` prevents the container from running as root.


## Part 7. Run an agent inside the container (10 points)

Before the container agent edits anything, return to the host, inspect the repository, and commit the reviewed `Dockerfile` and current notebook with the message:

```text
checkpoint: add container before container agent changes
```

Confirm that the tree is clean. Start the container again with the repository mounted at `/workspace`, launch Copilot CLI from `/workspace`, confirm the sandbox and repository boundary, and give it this task:

```text
Extend src/text_stats.py with an optional --top N argument that reports the N
most frequent words, case-insensitively, with deterministic tie-breaking.
Update the unittest tests and README.md. Do not add external dependencies. Do
not commit or push. First inspect the existing project and explain your plan.
After editing, show the diff and run all tests.
```

After the agent finishes:

1. Exit the container.
2. Confirm on the host that the bind-mounted changes persisted.
3. Inspect the entire diff and run the tests on the host.
4. Try the new option with `sample.txt` and check the result.
5. Commit the reviewed change as `feat: add top-words option from container agent`.
6. Push the branch.

### Part 7 laboratory record

Provide selected evidence that Copilot ran inside the container, changed the bind-mounted repository, and ran tests. Include the checkpoint and post-agent commit identifiers.

**Answer**

- Pre-agent checkpoint commit: `db1dd6f` ("checkpoint: add container before container agent changes")
- Post-agent reviewed commit: `8acb270` ("feat: add top-words option from container agent")
- Follow-up cleanup commit: `d131d1e` ("chore: remove stray file created by a shell redirection typo")

**Copilot running inside the container.** I started the container from the host with the repository bind-mounted, then launched Copilot from `/workspace`:

```text
appuser@<container-id>:/workspace$ copilot

  ╭─╮╭─╮
  ╰─╯╰─╯  Copilot v1.0.87 uses AI.
  █ ▘▝ █  Check for mistakes.

Folder /workspace has been added to trusted folders.

You must be logged in to send messages. Please run /login
Signed in successfully as <REDACTED-GITHUB-USERNAME>!
```

Because the previous container was started with `--rm`, its filesystem was discarded on exit, so the earlier `gh` and Copilot logins did not persist. I authenticated again inside the new container with `/login` and completed the device flow in my host browser. The trusted folder is `/workspace`, the mounted repository, so the agent's boundary is the project directory rather than a host path.

**Agent planning, tool use, and verification (excerpts).** I gave it the supplied Part 7 prompt unchanged. It inspected the project before editing:

```text
I'll inspect the repository state, src/text_stats.py, its tests, and README.md first...
Shell  git status --short && find . -maxdepth 2 -type f | sort
Read   text_stats.py (30 lines), README.md (33 lines), test_text_stats.py (62 lines)

The worktree is already dirty, including the requested files plus unrelated project
files. I'll inspect the existing diffs now so I can avoid overwriting user work; if
the target-file changes conflict with this request, I'll stop for guidance.
Shell  git --no-pager diff -- src/text_stats.py tests/test_text_stats.py README.md

The target-file diffs contain only line-ending normalization, not substantive logic,
so there is no conflicting user implementation to preserve. I'll keep the existing
output unchanged when --top is omitted; when supplied, I'll add a top array of
{"word": ..., "count": ...} entries, normalize with Unicode-aware casefold(), and
sort ties alphabetically by the normalized word for deterministic results.
```

This shows the `AGENTS.md` rule about stopping on unrelated changes working as intended: the agent noticed the dirty tree, investigated it, and correctly identified it as CRLF line-ending noise instead of editing over what it thought was user work.

It then edited the three files, ran the tests, fixed its own mistake, and caught an edge case:

```text
Edit   src/text_stats.py, tests/test_text_stats.py, README.md
Shell  python -m unittest discover -s tests

The first full test run found only incorrect expected character counts in the new
tests (the implementation counted the input lengths correctly). I also caught an edge
case before finalizing: a negative N would produce incorrect Python slice behavior,
so I'll reject negative values explicitly and correct the test fixtures.

Shell  python -m unittest discover -s tests          -> 6 tests passed
Shell  python src/text_stats.py sample.txt --top 3
Shell  git status --short
```

It reported no commit and no push, which matched the prompt's constraint.

**Bind-mounted changes persisted on the host.** After I exited Copilot and the container, the host working tree showed exactly the three expected files changed, which proves the container wrote through the bind mount onto my Windows disk:

```text
(base) PS C:\csci6032-hw2-batorgil7> git status
On branch agent-work
Changes not staged for commit:
        modified:   README.md
        modified:   src/text_stats.py
        modified:   tests/test_text_stats.py
```

**My own verification on the host.** I read the full diff, then ran the tests and the CLI myself:

```text
(base) PS C:\csci6032-hw2-batorgil7> python -m unittest discover -s tests -v
test_cli_reads_sample_file_and_prints_json ... ok
test_cli_reads_utf8_file ... ok
test_cli_reports_top_words ... ok
test_text_stats_counts_unicode_and_whitespace ... ok
test_text_stats_empty_text ... ok
test_text_stats_reports_top_words_case_insensitively ... ok
----------------------------------------------------------------------
Ran 6 tests in 0.592s

OK

(base) PS C:\csci6032-hw2-batorgil7> python src/text_stats.py sample.txt --top 3
{"lines": 5, "words": 40, "characters": 227, "top": [{"word": "the", "count": 4}, {"word": "and", "count": 2}, {"word": "i", "count": 2}]}

(base) PS C:\csci6032-hw2-batorgil7> python src/text_stats.py sample.txt --top 0
{"lines": 5, "words": 40, "characters": 227, "top": []}
```

I checked the result against `sample.txt` independently by counting the words myself. Folded to lowercase, `the` appears 4 times ("The" opens line 1, and "the" appears three more times), while `is`, `and`, and `i` each appear twice. The program returned `the`, `and`, `i`, which confirms both the case-insensitive folding and the alphabetical tie-break among the three words tied at 2: `and` before `i` before `is`. I also tried `--top 0`, which returns an empty list rather than failing or dumping every word.

**Correction I made, not the agent.** While reviewing the terminal afterward I noticed that a stray file named `t discover -s tests -v` had been committed in `8acb270`. It was not agent output. When I pasted a multi-line command block, PowerShell read a leading `>>` as an append redirection and wrote the `less` pager help text into a file with that name, and my `git add -A` then staged it. I removed it in `d131d1e` with `git rm`. I did not rewrite history, so the mistaken commit stays visible in the log and the final `HEAD` is clean.

Describe at least one difference between running the agent directly on the host and running it in the container. Which boundary protected the host, and which host files were intentionally exposed?

**Answer**

**Differences between the host run and the container run.** On the host in Part 4, Copilot ran as my own Windows user account. Local sandboxing was unavailable in Copilot CLI 1.0.87 on Windows, so nothing at the operating-system level stopped it from reading or writing outside the repository. The only protections were the trusted-folder setting, `AGENTS.md`, and my approval of each permission prompt. In the container, the agent ran as the non-root Linux user `appuser` (uid 1001) inside a separate filesystem whose only view of my computer was the single bind mount at `/workspace`. Even a command that tried to reach a host path would find nothing there, because those paths do not exist inside the container.

A second practical difference is that container state is disposable. Because I ran with `--rm`, everything outside `/workspace`, including the `gh` and Copilot credentials created inside the container, was destroyed when I exited, and I had to authenticate again in the next container. On the host, installs and logins persist in my user profile.

**Which boundary protected the host.** The container boundary did: Linux namespace isolation plus the single explicit bind mount. That is an enforced boundary, unlike `AGENTS.md`, which is only instructions an agent may or may not follow. The non-root user added a second layer, keeping the agent from installing packages or modifying system files even inside the container.

**Which host files were intentionally exposed.** Exactly one directory: the homework repository, mounted read/write at `/workspace`. That deliberately includes everything in it, the notebook, `README.md`, `sample.txt`, `src/`, `tests/`, `AGENTS.md`, `Dockerfile`, and the `.git` directory itself. So the container agent could in principle have altered my Git history, which is why the pre-agent checkpoint `db1dd6f` was committed and pushed to GitHub before the container agent ran, and why the prompt forbade committing and pushing. Everything else on the host stayed hidden: my home directory, `.ssh` keys, Chrome profile, the Docker socket, and every other folder on the C: drive.

One boundary the container did **not** provide is network isolation. The container had ordinary internet access, and once I ran `gh auth login` inside it, that credential could reach any repository my GitHub account can, not just this homework repo. The container limited which host *files* the agent could touch, but not its network reach. That is why review and pushing stayed on the host, where I inspected the diff myself before committing.

## Part 8. Add browser tools on the host (5 points)

Return to the **host** Copilot installation for this part. Do not attempt to share your Chrome profile with the development container.

GitHub Copilot CLI includes Playwright browser tools. To allow an agent to operate a tab in your existing authenticated Chrome session, install the instructor-approved [Playwright MCP Bridge extension](https://github.com/microsoft/playwright/tree/main/packages/extension) and configure a project-level Playwright MCP server in extension mode by following the current [Playwright MCP documentation](https://github.com/microsoft/playwright-mcp) and [GitHub Copilot MCP documentation](https://docs.github.com/en/copilot/how-tos/copilot-cli/customize-copilot/add-mcp-servers).

Important boundaries:

- Install only the extension linked by the instructor.
- Review the permissions Chrome requests.
- Connect only the Blackboard tab needed for this assignment.
- Keep the default connection approval; do not store an extension connection token in the repository.
- Project MCP configuration is executable configuration. Read it before trusting the folder.
- Test the browser connection with a read-only action, such as asking the agent to report the title of the selected tab.

Do not allow the agent to submit anything during this test.

### Part 8 laboratory record

Record the extension name and source, the location of the MCP configuration, and the result of the read-only browser test. Do not paste cookies, tokens, Blackboard contents, grades, or personal information.

**Answer**

- **Extension:** Playwright MCP Bridge extension ("Playwright Extension"), the instructor-approved one from the Playwright repository: https://github.com/microsoft/playwright/tree/main/packages/extension. I installed no other extension and reviewed the permissions Chrome requested before accepting.
- **MCP configuration location:** `.mcp.json` at the root of this repository. Copilot CLI searches from the working directory up to the repository root and reads `.mcp.json` as project-level configuration, so it applies only when I run Copilot inside this repo.

The whole file, which contains no token, credential, or personal path:

```json
{
  "mcpServers": {
    "playwright": {
      "type": "local",
      "command": "npx",
      "args": [
        "@playwright/mcp@latest",
        "--extension"
      ]
    }
  }
}
```

I read this before trusting the folder, since project MCP configuration is executable configuration. It launches the Playwright MCP server with `--extension`, which is the flag for connecting to an already-running Chrome through the bridge extension instead of launching a fresh browser. I deliberately left out a `tools` key so the default permission prompts remain in place rather than pre-approving browser tools, and there is no extension connection token in the repository.

**Read-only browser test.** I opened my Blackboard tab in Chrome, authenticated to Blackboard personally, then started `copilot` from the repository root on the host:

```text
MCP Servers:

  • github-mcp-server — not connected
  • playwright — connected

> What is the title of the currently selected browser tab?

Manage tabs (MCP: playwright) · action: "list"

### Result

The currently selected browser tab title is: Blackboard Ultra
```

The only browser action was `list`, which reads the open tabs. I did not let the agent click, type, navigate, or submit anything during this test. The `github-mcp-server` entry is an unrelated built-in server that failed to start; it does not affect the Playwright connection.

What additional authority did connecting the browser give the agent?

**Answer**

Connecting the browser gave the agent my authenticated identity on every site that Chrome session is logged into. Up to this point the agent could edit files in the repository and, inside the container, reach GitHub with a token I had created. Now it can act inside an already-logged-in browser, so it never needs my password and never sees a credential: it simply inherits the session I established. Anything I can do by clicking in that tab, the agent can now attempt, including reading page content and pressing buttons that cause irreversible actions such as submitting an assignment.

This authority is also broader than the one directory the container mount allowed. The bind mount limited the agent to the homework repository, but a browser session is not scoped to one page. Through the bridge extension it can enumerate tabs and read their titles and URLs, which is exactly what my read-only test did, so anything open in that Chrome profile is potentially visible to it.

That is why this part belongs on the host with the default connection approval left in place, why only the Blackboard tab needed for this assignment should be connected, and why the submission skill must stop and ask again immediately before the final submit. A file edit can be reverted with Git, but a Blackboard submission cannot be undone by `git restore`.

## Part 9. Create a Blackboard submission skill (5 points)

Create a project skill at:

```text
.github/skills/blackboard-submission/SKILL.md
```

You may use Copilot to draft the skill. The `SKILL.md` file must have valid YAML frontmatter containing a lowercase hyphenated `name` and a precise `description`. Do not include `allowed-tools: "*"`, and do not preapprove shell or browser tools. The point is to preserve visible permission boundaries.

The skill must implement this workflow:

1. Confirm that it is operating in the expected homework repository.
2. Run a preflight that checks the current branch, `git status`, recent commits, expected remote URL, and required files.
3. Stop if the tree is not clean, required artifacts are absent, or unresolved secrets/private data are apparent.
4. Confirm that the reviewed branch has been pushed.
5. Prepare `csci6032-hw2-<your-github-username>.tar.gz` from the committed `HEAD` without including `.git`, credentials, caches, or unrelated files.
6. List the archive contents and show the exact notebook, archive, repository URL, and submission text it proposes to use.
7. Support a **dry run** that performs every possible check without opening Blackboard or submitting.
8. Ask before opening or controlling the Blackboard tab.
9. Require the student to authenticate personally; never request, read, store, type, or expose credentials.
10. Navigate only to this homework's submission page and stage the required files and repository URL.
11. Stop immediately before the final, irreversible submission action and show the student exactly what will be submitted.
12. Require explicit confirmation at that point. A prior general approval is not sufficient.
13. After confirmation, complete the submission, verify the confirmation page or receipt, and report the result.
14. Do not commit Blackboard screenshots, receipts, browser data, or personal information to the public repository.

Review the current [GitHub agent skills documentation](https://docs.github.com/en/copilot/how-tos/copilot-cli/customize-copilot/add-skills). Reload and inspect the skill with `/skills reload`, `/skills list`, and `/skills info blackboard-submission`.

Commit the skill and final notebook changes as:

```text
feat: add Blackboard submission skill
```

Push the branch. Then return to your repository's default branch, merge
`agent-work` **without squashing**, and push the default branch. Resolve and
review any conflict rather than asking Git to discard one side. Run the tests
again after the merge. The default branch, working tree, and remote must agree
before you create the submission archive.

### Part 9 laboratory record

Paste your complete `SKILL.md` here as a fenced code block.

**Answer**

Skill location and registration, confirmed with `/skills reload`, `/skills list`, and `/skills info blackboard-submission`:

```text
Skills reloaded. Found 5 skills.

Available Skills

project:
  - blackboard-submission
    Safely prepare, review, and submit the CSCI 6032 homework archive to Blackboard
    only after a dry run, explicit human approval, and verification of the final
    confirmation page.

Skill: blackboard-submission
Source: Project
Location: <REPO-ROOT>\.github\skills\blackboard-submission\SKILL.md
```

Complete `SKILL.md`:

````markdown
---
name: blackboard-submission
description: Safely prepare, review, and submit the CSCI 6032 homework archive to Blackboard only after a dry run, explicit human approval, and verification of the final confirmation page.
---

# Blackboard submission skill

Use this skill only for this homework repository and only for the Blackboard submission flow described in the assignment. The goal is to protect the student from accidental submission of the wrong archive, unpublished work, or private data while still allowing a controlled submission. Before submitting check if all homework questions are answered and are correct.

## Required repository and safety checks

1. Confirm the working directory is the expected homework repository.
   - Verify the repo root contains the required project files: `CSCI6032_hw2.ipynb`, `README.md`, `AGENTS.md`, `sample.txt`, `Dockerfile`, `src/text_stats.py`, `tests/test_text_stats.py`, and `.github/skills/blackboard-submission/SKILL.md`.
   - Confirm the remote URL matches the expected public GitHub repository for this student and assignment.
   - Confirm the current branch is the reviewed branch, not a detached HEAD or unrelated worktree.

2. Run a preflight before any browser action.
   - Check `git status --short` and stop if the tree is not clean.
   - Check recent commits (`git log --oneline -n 10`) to confirm the required checkpoints and the final reviewed state are present.
   - Confirm the branch has been pushed to the remote and is up to date with the remote tracking branch.
   - Inspect the repo for obvious secrets or personal data: `.env`, credentials, browser profile folders, tokens, SSH keys, Docker socket paths, or local MCP config containing private paths or extension tokens.
   - If any of the above fail, stop and report the problem clearly. Do not continue.

3. Require a clean, reviewed submission state.
   - The archive must be created from the committed `HEAD` only.
   - Do not include `.git`, caches, credentials, browser state, or unrelated files.
   - Do not include secrets, private information, or personal paths in the notebook or archive contents.

## Dry-run workflow

4. Support dry-run mode first, and treat dry run as the default.
   - Assume dry run unless the student explicitly asks for a real submission in this invocation.
   - Run all possible safety checks and output the exact work that would be packaged.
   - Show the proposed archive name, archive contents, repository URL, notebook path, and exact Blackboard submission text.
   - Do not open Blackboard, do not click anything, and do not submit in dry-run mode.
   - Exit with explicit findings if any check fails.

5. Prepare the submission artifact.
   - Use the committed `HEAD` state only.
   - Build the archive with `git archive`, which reads from the commit and therefore cannot include `.git`, uncommitted edits, or ignored files:

     ```bash
     git archive --format=tar.gz --prefix=csci6032-hw2-<github-username>/ \
       -o csci6032-hw2-<github-username>.tar.gz HEAD
     ```

   - Do not build the archive by taring the working directory, which would capture `.git`, caches, and untracked files.
   - List the contents and show them before any browser action:

     ```bash
     tar -tzf csci6032-hw2-<github-username>.tar.gz
     ```

   - Leave the archive untracked. Never stage or commit it; `.gitignore` excludes `*.tar.gz`.

6. Display the exact submission payload.
   - Report the exact notebook file to be submitted.
   - Report the exact archive file to be uploaded or attached.
   - Report the repository URL that will be included in Blackboard.
   - Report the exact submission text to be entered in Blackboard.
   - If the student has not reviewed and approved the final payload, stop.

## Browser and Blackboard safety rules

7. Ask before opening or controlling the Blackboard tab.
   - Before any browser operation, explain the action and ask for explicit permission.
   - Never open a browser tab or interact with Blackboard in the background without the student's approval.

8. Require personal authentication only.
   - Do not request, read, store, type, or expose credentials.
   - The student must authenticate to Blackboard personally in a browser session.
   - The skill must never ask for tokens, passwords, cookies, or device codes.

9. Navigate only to the homework submission page.
   - Use the existing authenticated Blackboard tab only for this assignment.
   - Do not browse unrelated pages, course content, or student portal pages.
   - If the target page is not available or the student is not authenticated, stop and instruct the student to complete that step.

10. Stage only the required files and repository URL.
   - Fill in the correct repository URL.
   - Attach or stage the required notebook and archive only.
   - Review the final fields before any irreversible action.

11. Stop immediately before the final submission action.
   - Present the exact final payload, including archive name, notebook, repository URL, and Blackboard text.
   - Show the student the exact action that would happen if they confirm.
   - Do not click the final submit button automatically unless the student has explicitly confirmed that exact payload.

12. Require explicit confirmation at that point.
   - A general prior approval is not sufficient.
   - The student must explicitly confirm the final submission action immediately before submission.
   - If there is any doubt, stop and return to the review step.

## Final submission and verification

13. After explicit confirmation, complete the final submission action, verify the receipt or confirmation page, and report the result.
   - Capture the confirmation page or receipt text only as needed to confirm success.
   - Do not store browser data, screenshots, or receipts in the repository.
   - If the automation cannot safely perform the final click, stop and ask the student to submit manually while explaining the limitation.

14. Do not commit Blackboard artifacts to the public repository.
   - Never add screenshots, receipts, browser data, cookies, session data, or personal information to commits.
   - Keep all Blackboard confirmation evidence out of version control and out of the repository.
   - Any browser or submission artifacts must remain local-only and be explicitly excluded from the public repo.

## Example operating flow

The host shell for this repository is PowerShell on Windows, so prefer Git commands
and `Get-ChildItem` over Unix-only utilities such as `ls -1`.

```bash
# 1) Preflight checks
git rev-parse --show-toplevel
git remote -v
git status --short
git log --oneline -n 10
git ls-files

# 2) Dry run (the default mode)
# Perform checks, build the archive from HEAD with git archive, list its contents,
# print archive file name, repo URL, notebook file, and final Blackboard text.
# Do not open Blackboard.

# 3) Only after student approval and authentication
# Navigate to the homework submission page and stage the required files.
# Display the exact final payload.
# Ask again for explicit confirmation just before final submit.

# 4) After final confirmation
# Complete the final submit.
# Read the confirmation page or receipt.
# Report the final result without committing any browser artifacts.
```

## Success criteria

The skill is successful only if:

- the repo is verified and safe,
- the branch is reviewed and pushed,
- the archive is prepared from the committed `HEAD`,
- the dry run is complete,
- the student personally authenticates,
- the final submission step is confirmed immediately before it happens,
- and the confirmation page is verified without storing Blackboard artifacts in the repository.
````

The frontmatter has a lowercase hyphenated `name` and a description that states both what the skill does and the conditions under which it may submit. There is no `allowed-tools: "*"` and no preapproved shell or browser tools, so every shell command and every browser action still raises its own permission prompt.

One note on how the skill was created: Copilot's first attempt to write the file failed with "Parent directory does not exist," because `.github/skills/blackboard-submission/` did not exist yet. It explained the error instead of ignoring it, created the directory, and then wrote the file, which is the behavior `AGENTS.md` asks for.

Describe one material change you made after reviewing the agent's draft of the skill.

**Answer**

The most important change I made was replacing a vague instruction with a command that is safe by construction. The agent's draft said only to "build `csci6032-hw2-<github-username>.tar.gz` from the repo root, excluding `.git` and unrelated files." That states the goal but leaves the method to whichever agent runs the skill later, and the obvious method, taring the working directory, would do the opposite of what the assignment requires: it would sweep in `.git`, untracked scratch files, `__pycache__`, and any uncommitted edits. I had already created one stray file by accident during Part 7, so this was a realistic risk, not a hypothetical one.

I rewrote requirement 5 to specify `git archive`:

```bash
git archive --format=tar.gz --prefix=csci6032-hw2-<github-username>/ \
  -o csci6032-hw2-<github-username>.tar.gz HEAD
```

`git archive` reads directly from the named commit rather than from the working tree, so `.git`, uncommitted changes, and ignored files cannot end up in the archive at all. The requirement is enforced by the tool instead of relying on an agent remembering an exclusion list. I also added the `tar -tzf` listing command, an explicit warning against taring the working directory, and a rule that the archive stays untracked, plus `*.tar.gz` in `.gitignore` so it cannot be committed by a careless `git add -A`.

I made three smaller revisions as well:

- **Dry run is now the default.** The draft supported a dry run but never said which mode a bare invocation gets. I made dry run the assumed mode unless I explicitly ask for a real submission, so simply running `/blackboard-submission` cannot reach a submit button.
- **Fixed a shell mismatch.** The example flow used `ls -1`, which does not exist in PowerShell on my host, so an agent following the skill literally would hit an error. I replaced it with `git ls-files` and noted that the host shell is PowerShell.
- **Kept permission boundaries visible.** I confirmed there is no `allowed-tools` key, so shell and browser tools still prompt individually rather than being preapproved.

Why must the skill ask again immediately before the final submission action?

**Answer**

Because clicking submit is the one step in this entire assignment that Git cannot undo. Every other action has a recovery path: a bad edit is fixed with `git restore`, a bad commit with `git revert`, a broken container by exiting and starting a new one. A Blackboard submission goes to a system I do not control, it is timestamped, and my instructor may see it immediately. There is no checkpoint to return to.

The confirmation also has to happen at that exact moment because what I approved earlier is not what would be submitted now. When I approve connecting the browser, or approve a dry run, I am approving a plan. Between then and the final click the archive could have been rebuilt, the wrong file could have been staged, the page could have changed, or the agent could have misread which assignment it was on. Approval of a plan is not approval of the specific bytes now sitting in the upload field, so the skill has to show me the actual archive, the actual notebook, the actual repository URL, and the actual submission text and ask again about that payload.

There is also a difference between authority and intent. Once the browser is connected, the agent already has the technical authority to click submit whenever it likes; nothing at the operating-system level stops it. So the last checkpoint cannot be a permission boundary, only a deliberate pause written into the skill. That pause is what keeps a human, rather than an agent's judgment about when it is finished, in charge of the irreversible step.

## Part 10. Dry-run and use the submission skill

Invoke `/blackboard-submission` in dry-run mode first. Resolve every reported problem. Inspect the archive contents yourself, confirm the repository is public and up to date, and confirm that no secret or private information is included.

Then invoke the skill for the real submission. Personally authenticate to Blackboard. Let the skill prepare the submission, inspect the staged files and text, and provide explicit confirmation only when they are correct. Verify the resulting Blackboard confirmation.

Do not modify the committed notebook afterward merely to record that submission succeeded; doing so would make the submitted archive differ from the repository checkpoint. Instead, enter this short statement in Blackboard alongside the repository URL:

```text
I used my blackboard-submission skill, reviewed the staged artifacts, explicitly
approved the final submission action, and verified Blackboard's confirmation.
```

If the automation cannot safely complete the final click, perform that click yourself, verify the result, and explain the limitation in the Blackboard comment. Safe human completion is preferable to bypassing a security boundary.

## Required Git history and repository contents

Your public repository must show, at minimum, these ordered checkpoints:

1. `checkpoint: repository before agent changes`
2. `docs: add agent safety instructions`
3. `feat: add agent-generated text statistics tool`
4. `checkpoint: add container before container agent changes`
5. `feat: add top-words option from container agent`
6. `feat: add Blackboard submission skill`

Equivalent additional commits are welcome. Do not squash these checkpoints.

Required contents:

```text
CSCI6032_hw2.ipynb
README.md
AGENTS.md
sample.txt
Dockerfile
src/
  text_stats.py
tests/
  test_text_stats.py
.github/
  skills/
    blackboard-submission/
      SKILL.md
```

Your project-level MCP configuration may be committed only if it contains no secret, personal path, browser data, or extension token. Otherwise, document its structure in the notebook and exclude it from the repository.

## Final reflection (5 points)

Answer each question in a short paragraph.

1. Where did Git provide protection, and where did it not?
2. How did the host sandbox and Docker container differ as boundaries?
3. Which agent output required the most human judgment?
4. What did the agent do that saved you time?
5. What would you change before allowing an agent to work on a valuable research repository?

**Answer**

**1. Where did Git provide protection, and where did it not?**

Git protected everything inside the repository's tracked history. Before each agent phase I committed and pushed a checkpoint (`639b591`, `db1dd6f`), so at every point I could compare the agent's work against a named starting state with `git diff` and discard it if needed. The pushed copy on GitHub also meant the checkpoint survived independently of my laptop. Git did not protect me from the things that never entered a commit, or that entered one by accident. It did not stop me from committing a stray file I created with a PowerShell redirection typo, and once that file was in `8acb270` it stayed in history even after I removed it in `d131d1e`. Git also offered nothing at all for the parts of this assignment that live outside the repository: it cannot undo a `gh` login, a Docker image, or a Blackboard submission. And because the whole repository including `.git` was mounted into the container, Git's own history was inside the blast radius rather than protecting it.

**2. How did the host sandbox and Docker container differ as boundaries?**

The host sandbox did not exist. `/sandbox` was not a recognized command in Copilot CLI 1.0.87 on Windows, so on the host the only things standing between the agent and the rest of my drive were the trusted-folder setting, the instructions in `AGENTS.md`, and my willingness to read each permission prompt before approving it. Those are conventions, not barriers: a text file cannot stop anything. The container was an actual boundary, enforced by the operating system. The agent ran as a non-root user in a separate filesystem, and the only part of my computer visible to it was the single bind mount at `/workspace`. Unrelated host paths did not merely fail a check, they did not exist. The container boundary was also narrower than I first assumed: it constrained files, not network access, so a `gh` token created inside it could still reach every repository my GitHub account can.

**3. Which agent output required the most human judgment?**

The `--top N` tie-breaking, because the tests the agent wrote could not tell me whether the behavior was right. Six tests passed, but they passed against expectations the agent had chosen itself, so agreeing with them proved only self-consistency. I had to count the words in `sample.txt` myself to learn that `is`, `and`, and `i` all appear twice, and only then could I confirm that returning `and` before `i` was a real alphabetical tie-break rather than accidental dictionary ordering. The Part 3 answer needed the same kind of judgment in the other direction: a draft answer claimed unrelated host locations were "not writable by policy" when sandboxing was unavailable, which is precisely backwards, and a plausible-sounding sentence like that is harder to catch than a failing test.

**4. What did the agent do that saved me time?**

The mechanical, well-specified work. It produced the `text_stats.py` CLI, the unittest suite, and the README usage sections faster than I would have typed them, and the Dockerfile saved me a lot of searching, particularly the `gh` apt keyring setup and the `safe.directory` line that prevents Git ownership errors on a bind-mounted repository. It also caught two things I would probably have missed: that a negative `--top` value would silently produce wrong output through Python's slice behavior, and that the seven "modified" files in the container were CRLF line-ending noise rather than real edits, which it diagnosed by reading the diff instead of editing over what it thought was my work.

**5. What would you change before allowing an agent to work on a valuable research repository?**

I would keep the container and tighten what goes into it. I would mount only the subdirectory the task actually needs rather than the whole repository, and mount data read-only, because in this assignment the agent had write access to `.git` itself. I would not authenticate `gh` inside the agent's environment at all; pushing is a human step on the host after I have read the diff, and a session-scoped or repository-scoped token would be better than one with my full account reach. I would restrict network access rather than leaving the container with open internet. On the process side, I would branch for every agent task and never let an agent work on the default branch, require tests written or at least reviewed by me instead of tests the agent wrote to match its own implementation, and add a `.gitattributes` file so line endings are normalized and `git status` stays meaningful across Windows and Linux. Most of all I would keep the habit this assignment enforced: a pushed checkpoint before delegating anything, and my own verification rather than the agent's claim of success.

## Grading summary

| Component | Points |
|---|---:|
| Access and preparation | 5 |
| Host Copilot installation and Hello World | 10 |
| Repository and initial checkpoint | 10 |
| Agent instructions and host boundary | 10 |
| Host-agent code generation and review | 15 |
| Docker installation | 10 |
| Linux agent development image | 15 |
| Agent work inside the container | 10 |
| Browser tools | 5 |
| Submission skill | 5 |
| Final reflection | 5 |
| **Total** | **100** |

The real Blackboard submission is required for the homework to be received. If browser automation fails after a documented good-faith attempt, submit manually and document the limitation in the Blackboard comment.

## Documentation references

These links were checked when the assignment was prepared. Tools change quickly; use the current official page when its interface differs from this notebook.

- [Installing GitHub Copilot CLI](https://docs.github.com/en/copilot/how-tos/copilot-cli/set-up-copilot-cli/install-copilot-cli)
- [Authenticating GitHub Copilot CLI](https://docs.github.com/en/copilot/how-tos/copilot-cli/set-up-copilot-cli/authenticate-copilot-cli)
- [Using local sandboxing](https://docs.github.com/en/copilot/how-tos/cloud-and-local-sandboxes/using-local-sandboxing)
- [Allowing and denying Copilot CLI tools](https://docs.github.com/en/copilot/how-tos/copilot-cli/use-copilot-cli/allowing-tools)
- [Adding agent skills](https://docs.github.com/en/copilot/how-tos/copilot-cli/customize-copilot/add-skills)
- [Adding MCP servers](https://docs.github.com/en/copilot/how-tos/copilot-cli/customize-copilot/add-mcp-servers)
- [Docker Desktop](https://docs.docker.com/desktop/)
- [Docker Engine installation](https://docs.docker.com/engine/install/)
- [Docker bind mounts](https://docs.docker.com/engine/storage/bind-mounts/)
- [Playwright MCP](https://github.com/microsoft/playwright-mcp)
- [Playwright MCP Bridge extension](https://github.com/microsoft/playwright/tree/main/packages/extension)